## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), con m = max_j z_j. Esta reescritura no altera el resultado matemático pero evita desbordamientos cuando los logits son grandes o T pequeño. Con los logits z = (2.0, 1.0, 0.5, 0.2, −1.0, −3.0) para los tokens t0…t5, las distribuciones y las entropías en bits son:

| T | t0 | t1 | t2 | t3 | t4 | t5 | H (bits) |
|---|---|---|---|---|---|---|---|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Cada fila suma 1.0 (verificado en el notebook). El 0.0000 de t5 con T = 0.5 es efecto del redondeo a cuatro decimales: su valor exacto es 3.7367e-05. Se aprecia el efecto esperado de la temperatura: al bajarla de 2 a 0.5 la masa se concentra en t0 (0.3583 → 0.5516 → 0.8231) y la entropía cae monótonamente (2.2888 → 1.7998 → 0.9147 bits); al subirla, la distribución se aplana y la entropía crece.

In [1]:
# T1: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura (NumPy, estable) + entropía en bits.
Tarea B — Temperatura, top-p y entropía · MMIA 6013
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# 1) Softmax con temperatura, numéricamente estable (se resta el máximo)
# ----------------------------------------------------------------------
def softmax_con_temperatura(logits, T):
    """p_i = exp(z_i/T) / sum_j exp(z_j/T), restando el máximo para estabilidad."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - np.max(z)          # estabilidad numérica: evita overflow de exp
    e = np.exp(z)
    return e / np.sum(e)


def entropia_bits(p):
    """H(p) = -sum p_i log2 p_i, en bits (se ignoran p_i = 0 por convención 0*log2(0)=0)."""
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return float(-np.sum(p[mask] * np.log2(p[mask])))


# ----------------------------------------------------------------------
# 2) Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]

distribuciones = {T: softmax_con_temperatura(logits, T) for T in temperaturas}
entropias = {T: entropia_bits(distribuciones[T]) for T in temperaturas}

# ----------------------------------------------------------------------
# 3) Tabla con cuatro decimales (distribuciones + entropías)
# ----------------------------------------------------------------------
filas = []
for T in temperaturas:
    filas.append([f"{T:.1f}"] + [f"{p:.4f}" for p in distribuciones[T]]
                 + [f"{entropias[T]:.4f}"])
tabla = pd.DataFrame(filas, columns=["T"] + tokens + ["H_bits"])
print("Tabla (4 decimales):")
print(tabla.to_string(index=False))

# Verificación: cada fila de probabilidades suma 1.0
sumas = {f"T={T}": float(np.sum(distribuciones[T])) for T in temperaturas}
print("\nSumas de cada fila de probabilidades:", sumas)

# ----------------------------------------------------------------------
# 4) Figura: distribuciones para cada temperatura
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(tokens))
ancho = 0.25
for i, T in enumerate(temperaturas):
    ax.bar(x + (i - 1) * ancho, distribuciones[T], ancho, label=f"T = {T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("p_i")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura (logits 2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
fig.tight_layout()
fig.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {f"T={T}": distribuciones[T].tolist() for T in temperaturas},
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "suma_probabilidades_por_fila": sumas,
    "tabla_4_decimales": {
        "columnas": ["T"] + tokens + ["H_bits"],
        "filas": filas,
    },
}
Path("resultados.json").write_text(
    json.dumps(resultados, indent=2, ensure_ascii=False), encoding="utf-8"
)

# ----------------------------------------------------------------------
# 6) Resumen
# ----------------------------------------------------------------------
print("\nResumen T1:")
for T in temperaturas:
    print(f"  T={T}: p={np.round(distribuciones[T], 4).tolist()}  H={entropias[T]:.4f} bits")
print("Archivos generados: resultados.json, t1_softmax_temperatura.png")


Tabla (4 decimales):
  T     t0     t1     t2     t3     t4     t5 H_bits
0.5 0.8231 0.1114 0.0410 0.0225 0.0020 0.0000 0.9147
1.0 0.5516 0.2029 0.1231 0.0912 0.0275 0.0037 1.7998
2.0 0.3583 0.2173 0.1693 0.1457 0.0800 0.0294 2.2888

Sumas de cada fila de probabilidades: {'T=0.5': 1.0, 'T=1.0': 1.0, 'T=2.0': 1.0}

Resumen T1:
  T=0.5: p=[0.8231, 0.1114, 0.041, 0.0225, 0.002, 0.0]  H=0.9147 bits
  T=1.0: p=[0.5516, 0.2029, 0.1231, 0.0912, 0.0275, 0.0037]  H=1.7998 bits
  T=2.0: p=[0.3583, 0.2173, 0.1693, 0.1457, 0.08, 0.0294]  H=2.2888 bits
Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se ordenan los tokens por probabilidad descendente — t0, t1, t2, t3, t4, t5, orden que coincide con el índice porque los logits son decrecientes — y se acumula: 0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000. Con p = 0.9, el conjunto {t0, t1, t2} acumula solo 0.8776 (< 0.9), de modo que el conjunto mínimo que alcanza el umbral es {t0, t1, t2, t3}, con probabilidad acumulada 0.9688. Quedan descartados t4 (0.0275) y t5 (0.0037), es decir, una masa de 0.0312 que se redistribuye proporcionalmente entre los supervivientes al renormalizar.

| token | prob (T=1) | acumulada | ¿superviviente? | prob renormalizada |
|---|---|---|---|---|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | — |
| t5 | 0.0037 | 1.0000 | no | — |

La distribución renormalizada es t0 = 0.5694, t1 = 0.2095, t2 = 0.1270, t3 = 0.0941 (suma 1.0). Cada superviviente aumenta ligeramente su probabilidad respecto de la original (por ejemplo, t0 pasa de 0.5516 a 0.5694) como consecuencia de la renormalización tras descartar la masa 0.0312.

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==================================================================
# T2: Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución
#     de T = 1 obtenida en T1 (Parte 1).
# Criterio del enunciado: ordenar por probabilidad descendente,
# conservar el conjunto MÍNIMO cuya probabilidad acumulada ALCANZA
# p = 0.9 y renormalizar las probabilidades de los supervivientes.
# ==================================================================

P = 0.9
TOKENS_ENUNCIADO = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)


def softmax_T(z, T):
    """Softmax con temperatura, numéricamente estable (resta el máximo)."""
    z = np.asarray(z, dtype=float)
    z_stable = (z - z.max()) / T
    e = np.exp(z_stable)
    return e / e.sum()


# ------------------------------------------------------------------
# 1) Distribución de entrada: la de T = 1 de la subtarea T1
# ------------------------------------------------------------------
ruta_t1 = Path("entrada/T1/resultados.json")
if ruta_t1.exists():
    with open(ruta_t1, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    probs_t1 = np.array(t1["distribuciones"]["T=1.0"], dtype=float)
    tokens = list(t1["tokens"])
    fuente = str(ruta_t1)
else:
    # Respaldo: recalcular con la softmax estable del enunciado
    probs_t1 = softmax_T(LOGITS, 1.0)
    tokens = list(TOKENS_ENUNCIADO)
    fuente = "recalculada con softmax estable (T=1)"

# Verificación de consistencia con la softmax T=1
assert np.allclose(probs_t1, softmax_T(LOGITS, 1.0)), (
    "La distribución de T1 no coincide con la softmax T=1 recalculada"
)

# ------------------------------------------------------------------
# 2) Ordenar tokens por probabilidad descendente
# ------------------------------------------------------------------
orden = np.argsort(-probs_t1, kind="stable")
tokens_ord = [tokens[i] for i in orden]
probs_ord = probs_t1[orden]

# ------------------------------------------------------------------
# 3) Probabilidad acumulada y corte top-p:
#    conjunto mínimo (prefijo en orden descendente) con acumulada >= p
# ------------------------------------------------------------------
cum = np.cumsum(probs_ord)
k = int(np.searchsorted(cum, P, side="left"))  # primer índice con cum >= p
n_sup = k + 1

tokens_sup = tokens_ord[:n_sup]
probs_sup = probs_ord[:n_sup]
cum_sup = cum[:n_sup]
acumulada_corte = float(cum[k])  # prob. acumulada del conjunto superviviente

tokens_desc = tokens_ord[n_sup:]
probs_desc = probs_ord[n_sup:]

# ------------------------------------------------------------------
# 4) Renormalizar las probabilidades de los supervivientes
# ------------------------------------------------------------------
probs_renorm = probs_sup / acumulada_corte
suma_renorm = float(probs_renorm.sum())

# ------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T2",
    "descripcion": (
        "Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1 "
        "de la Parte 1: conjunto mínimo de tokens cuya probabilidad acumulada "
        "alcanza p, seguido de renormalización."
    ),
    "p": P,
    "temperatura": 1.0,
    "fuente_distribucion": fuente,
    "tokens": tokens,
    "distribucion_T1_entrada": probs_t1.tolist(),
    "orden_descendente": tokens_ord,
    "probs_ordenadas": probs_ord.tolist(),
    "prob_acumulada_ordenada": cum.tolist(),
    "tokens_supervivientes": tokens_sup,
    "prob_acumulada_antes_del_corte": acumulada_corte,
    "prob_acumulada_por_superviviente": [
        {"token": tok, "prob": float(pr), "acumulada": float(ac)}
        for tok, pr, ac in zip(tokens_sup, probs_sup, cum_sup)
    ],
    "distribucion_renormalizada": {
        tok: float(pr) for tok, pr in zip(tokens_sup, probs_renorm)
    },
    "distribucion_renormalizada_en_orden_original": {
        tok: float(probs_renorm[tokens_sup.index(tok)])
        for tok in tokens if tok in tokens_sup
    },
    "suma_distribucion_renormalizada": suma_renorm,
    "tokens_descartados": tokens_desc,
    "probs_descartados": probs_desc.tolist(),
    "masa_descartada": float(probs_desc.sum()),
    "tabla_4_decimales": {
        "columnas": ["token", "prob_T1", "acumulada", "superviviente",
                     "prob_renormalizada"],
        "filas": [
            [
                tok,
                f"{pr:.4f}",
                f"{ac:.4f}",
                "sí" if i < n_sup else "no",
                f"{probs_renorm[i]:.4f}" if i < n_sup else "-",
            ]
            for i, (tok, pr, ac) in enumerate(zip(tokens_ord, probs_ord, cum))
        ],
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# Figura: corte top-p y distribución renormalizada
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
x = np.arange(len(tokens_ord))
colores = ["#2a9d8f" if i < n_sup else "#d3d3d3" for i in range(len(tokens_ord))]
ax.bar(x, probs_ord, color=colores, edgecolor="black", linewidth=0.5,
       label="prob. T=1")
ax.plot(x, cum, "o--", color="#e76f51", label="acumulada")
ax.axhline(P, color="red", linestyle=":", linewidth=1.2, label=f"p = {P}")
ax.set_xticks(x)
ax.set_xticklabels(tokens_ord)
ax.set_ylim(0, 1.08)
ax.set_ylabel("probabilidad")
ax.set_title("Top-p (p=0.9) sobre la distribución T=1\nverde = supervivientes del núcleo")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(tokens_sup, probs_renorm, color="#2a9d8f", edgecolor="black", linewidth=0.5)
for i, pr in enumerate(probs_renorm):
    ax.text(i, pr + 0.012, f"{pr:.4f}", ha="center", fontsize=9)
ax.set_ylim(0, max(probs_renorm) * 1.18)
ax.set_ylabel("probabilidad renormalizada")
ax.set_title(f"Distribución renormalizada (suma = {suma_renorm:.6f})")

fig.tight_layout()
fig.savefig("T2_top_p.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# Resumen
# ------------------------------------------------------------------
print("=== T2: Top-p (nucleus sampling) con p = 0.9 sobre la distribución T=1 ===")
print(f"Fuente de la distribución: {fuente}")
print("Distribución T=1 de entrada:")
for tok, pr in zip(tokens, probs_t1):
    print(f"  {tok}: {pr:.6f}")
print("Orden descendente con probabilidad acumulada:")
for tok, pr, ac in zip(tokens_ord, probs_ord, cum):
    marca = "  <- corte aquí" if tok == tokens_sup[-1] else ""
    print(f"  {tok}: p={pr:.6f}  acum={ac:.6f}{marca}")
print(f"Tokens supervivientes (conjunto mínimo con acumulada >= {P}): {tokens_sup}")
print(f"Probabilidad acumulada antes del corte: {acumulada_corte:.6f}")
print("Distribución renormalizada de los supervivientes:")
for tok, pr in zip(tokens_sup, probs_renorm):
    print(f"  {tok}: {pr:.6f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.12f}")
print(f"Tokens descartados: {tokens_desc} (masa descartada = {probs_desc.sum():.6f})")
print("Archivos escritos: resultados.json, T2_top_p.png")


=== T2: Top-p (nucleus sampling) con p = 0.9 sobre la distribución T=1 ===
Fuente de la distribución: entrada/T1/resultados.json
Distribución T=1 de entrada:
  t0: 0.551623
  t1: 0.202931
  t2: 0.123084
  t3: 0.091183
  t4: 0.027464
  t5: 0.003717
Orden descendente con probabilidad acumulada:
  t0: p=0.551623  acum=0.551623
  t1: p=0.202931  acum=0.754553
  t2: p=0.123084  acum=0.877637
  t3: p=0.091183  acum=0.968820  <- corte aquí
  t4: p=0.027464  acum=0.996283
  t5: p=0.003717  acum=1.000000
Tokens supervivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada antes del corte: 0.968820
Distribución renormalizada de los supervivientes:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000000000
Tokens descartados: ['t4', 't5'] (masa descartada = 0.031180)
Archivos escritos: resultados.json, T2_top_p.png


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados fueron t0 = 5682, t1 = 2093, t2 = 1296 y t3 = 929 (suma 10 000). Las frecuencias relativas observadas se comparan con las probabilidades teóricas:

| token | prob. teórica | frec. observada | conteo |
|---|---|---|---|
| t0 | 0.5694 | 0.5682 | 5682 |
| t1 | 0.2095 | 0.2093 | 2093 |
| t2 | 0.1270 | 0.1296 | 1296 |
| t3 | 0.0941 | 0.0929 | 929 |

El diagrama de barras generado en la celda siguiente muestra ambas series prácticamente superpuestas. La divergencia KL de la distribución empírica respecto de la teórica es 5.0069e-05 bits (equivalente a 3.4705e-05 nats), un valor tan próximo a cero que confirma que las frecuencias observadas reproducen la distribución teórica dentro del ruido de muestreo esperable con n = 10 000.

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T3 — Comprobación empírica (Tarea B, MMIA 6013)
Toma 10 000 muestras de la distribución renormalizada de T2 (top-p p=0.9 sobre
la distribución de T=1) con numpy.random.default_rng(0), grafica las frecuencias
observadas frente a las probabilidades teóricas y calcula la divergencia KL
(en bits) de la distribución empírica respecto de la teórica.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Parámetros fijados por el enunciado
# ------------------------------------------------------------------
N_MUESTRAS = 10_000
SEMILLA = 0
P_TOP = 0.9
TEMPERATURA = 1.0

# ------------------------------------------------------------------
# 1) Cargar la distribución renormalizada de T2
# ------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    dist_dict = dict(t2["distribucion_renormalizada_en_orden_original"])
else:
    # Respaldo determinista: recalcular exactamente como en T2
    # (softmax con T=1, numéricamente estable, + top-p con p=0.9)
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p_t1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_t1)                      # descendente
    acum = np.cumsum(p_t1[orden])
    k = int(np.searchsorted(acum, P_TOP) + 1)      # conjunto mínimo que alcanza p
    supervivientes = np.sort(orden[:k])
    dist_dict = {f"t{i}": float(p_t1[i] / p_t1[supervivientes].sum())
                 for i in supervivientes}

tokens = list(dist_dict.keys())
p_teo = np.array([dist_dict[t] for t in tokens], dtype=float)
p_teo = p_teo / p_teo.sum()  # seguridad numérica (debe sumar 1)

# ------------------------------------------------------------------
# 2) Muestreo: 10 000 muestras con numpy.random.default_rng(0)
# ------------------------------------------------------------------
rng = np.random.default_rng(SEMILLA)
muestras = rng.choice(len(tokens), size=N_MUESTRAS, p=p_teo)

conteos = np.bincount(muestras, minlength=len(tokens))
freq_obs = conteos / N_MUESTRAS

# ------------------------------------------------------------------
# 3) Divergencia KL en bits: KL(empírica || teórica)
#    KL = Σ_i q_i * log2(q_i / p_i), con 0·log(0/p) := 0
# ------------------------------------------------------------------
mask = freq_obs > 0
kl_bits = float(np.sum(freq_obs[mask] * np.log2(freq_obs[mask] / p_teo[mask])))
kl_nats = float(kl_bits * np.log(2.0))

# ------------------------------------------------------------------
# 4) Diagrama de barras: frecuencias observadas vs. probabilidades teóricas
# ------------------------------------------------------------------
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, freq_obs, ancho, color="#1f77b4",
            label="Frecuencia observada (10 000 muestras)")
b2 = ax.bar(x + ancho / 2, p_teo, ancho, color="#ff7f0e",
            label="Probabilidad teórica (top-p p=0.9 renormalizada)")
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.006,
                f"{b.get_height():.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_title("T3: Frecuencias observadas vs. probabilidades teóricas\n"
             f"(muestreo de la distribución renormalizada top-p, T={TEMPERATURA}, p={P_TOP})")
ax.set_ylim(0, max(freq_obs.max(), p_teo.max()) * 1.18)
ax.legend()
plt.tight_layout()
plt.savefig("T3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 5) Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T3",
    "descripcion": (
        "Comprobación empírica: 10 000 muestras de la distribución renormalizada de T2 "
        "(top-p p=0.9 sobre la distribución de T=1) con numpy.random.default_rng(0); "
        "comparación de frecuencias observadas vs. probabilidades teóricas en un diagrama "
        "de barras y divergencia KL (en bits) de la empírica respecto de la teórica."
    ),
    "n_muestras": N_MUESTRAS,
    "semilla": SEMILLA,
    "generador": "numpy.random.default_rng(0)",
    "fuente_distribucion": "entrada/T2/resultados.json",
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teo)},
    "probabilidades_teoricas_lista": [float(p) for p in p_teo],
    "suma_probabilidades_teoricas": float(p_teo.sum()),
    "frecuencias_observadas_conteo": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas_conteo_lista": [int(c) for c in conteos],
    "frecuencias_observadas_relativas": {t: float(fr) for t, fr in zip(tokens, freq_obs)},
    "frecuencias_observadas_relativas_lista": [float(fr) for fr in freq_obs],
    "suma_conteos_observados": int(conteos.sum()),
    "suma_frecuencias_relativas": float(freq_obs.sum()),
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_nats": kl_nats,
    "figura": "T3_frecuencias_vs_teoricas.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# 6) Resumen
# ------------------------------------------------------------------
print("T3 — Comprobación empírica")
print(f"Muestras: {N_MUESTRAS} con numpy.random.default_rng({SEMILLA})")
print(f"Tokens (soporte top-p): {tokens}")
print("Probabilidades teóricas :", np.round(p_teo, 4))
print("Frecuencias observadas  :", np.round(freq_obs, 4))
print(f"Conteos observados      : {conteos.tolist()} (suma = {int(conteos.sum())})")
print(f"Divergencia KL(empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: T3_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


T3 — Comprobación empírica
Muestras: 10000 con numpy.random.default_rng(0)
Tokens (soporte top-p): ['t0', 't1', 't2', 't3']
Probabilidades teóricas : [0.5694 0.2095 0.127  0.0941]
Frecuencias observadas  : [0.5682 0.2093 0.1296 0.0929]
Conteos observados      : [5682, 2093, 1296, 929] (suma = 10000)
Divergencia KL(empírica || teórica) = 0.000050 bits
Figura guardada: T3_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0, el cociente z_i/T amplifica las diferencias entre logits: el término exp(z_i/T) del token de mayor logit (t0, con z = 2.0) crece mucho más rápido que el de los demás, de modo que toda la masa de probabilidad se concentra en t0 y la distribución tiende a una masa puntual (delta) en el argmax, con entropía que tiende a 0. La evidencia de la Parte 1 muestra esa tendencia: al reducir T de 2 a 1 y a 0.5, la probabilidad de t0 sube 0.3583 → 0.5516 → 0.8231 y la entropía baja 2.2888 → 1.7998 → 0.9147 bits, una caída monótona hacia cero.

Cuando T → ∞ ocurre lo contrario: z_i/T → 0 para todo token, todos los exp(z_i/T) tienden a 1 y la distribución tiende a la uniforme sobre los seis tokens, cuya entropía es el máximo posible para seis categorías (log2 6). La distribución de T = 2 es la más plana de las tres medidas (t0 = 0.3583 frente a t5 = 0.0294) y presenta la mayor entropía, 2.2888 bits.

T → 0 equivale a la decodificación voraz porque greedy selecciona siempre el token de mayor probabilidad — aquí t0 — y, en el límite T → 0, la softmax asigna probabilidad 1 a t0 y 0 al resto; muestrear de esa distribución devuelve t0 con certeza, exactamente el mismo resultado que tomar el argmax sobre los logits. La entropía lo cuantifica: mide la incertidumbre de la distribución y, según las cifras medidas, decrece hacia 0 (distribución determinista) conforme T disminuye, con 0.9147 bits ya en T = 0.5, donde t0 concentra 0.8231 de la masa.